# Demonstration of Vectorstore Usage for effecient retival form a large data

this example is currenlty made for html and pdf documents demonstrations

In [ ]:
%pip install -q langchain langchain-nvidia-ai-endpoints langchain_community

In [ ]:
%pip install -q unstructured faiss-cpu

In [ ]:
import os
from langchain_community.document_loaders import UnstructuredHTMLLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain.chains import RetrievalQA


In [ ]:
os.environ["NVIDIA_API_KEY"] = "nvapi-..."

## make sure to Upload documents into colab before loading them into the code

download some sample documents from somewhere and upload in this colab session
  




In [ ]:
%pip install "unstructured[pdf]"

In [ ]:
# 1. Load file

## Unstructured HTMLLoader: Good for html documents
## change the path if you have saved somewhere

# documents = UnstructuredHTMLLoader("/content/page_source_example.com_20251012_144902.html").load()
# documents = UnstructuredHTMLLoader("/content/page_source_docs_20251012_145425.html").load()
# documents = UnstructuredHTMLLoader("/content/page_source_soundcloud_20251012_152003.html").load()


from langchain.document_loaders import UnstructuredFileLoader
from langchain.document_loaders import ArxivLoader

## Unstructured File Loader: Good for arbitrary "probably good enough" loader
documents = UnstructuredFileLoader("llama2_paper.pdf").load()


/tmp/ipython-input-3167227679.py:15: LangChainDeprecationWarning: The class `UnstructuredFileLoader` was deprecated in LangChain 0.2.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-unstructured package and should be used instead. To use it run `pip install -U :class:`~langchain-unstructured` and import as `from :class:`~langchain_unstructured import UnstructuredLoader``.
  documents = UnstructuredFileLoader("llama2_paper.pdf").load()


In [ ]:
# 2. Split documents into manageable chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")

# 3. Initialize NVIDIA embeddings
embeddings = NVIDIAEmbeddings(
    model="nvidia/nv-embedqa-e5-v5",  # or another NVIDIA embedding model
    truncate="END"
)

# 4. Create FAISS vector store from chunks
print("Creating FAISS index...")
faiss_vectorstore = FAISS.from_documents(chunks, embeddings)

# 5. Set up retriever
retriever = faiss_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}  # Return top 4 most relevant chunks
)


Split into 644 chunks
Creating FAISS index...


In [ ]:

# 6. Initialize ChatNVIDIA for generation
llm = ChatNVIDIA(
    model="meta/llama-3.1-8b-instruct",  # or another NVIDIA chat model
    temperature=0.1,
    max_tokens=1000
)

# 7. Create RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    verbose=True
)

/tmp/ipython-input-445415549.py:2: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm = ChatNVIDIA(


In [ ]:
# 8. Query your HTML content
def ask_question(question):
    result = qa_chain({"query": question})

    print(f"\n**Question:** {question}")
    print(f"\n**Answer:** {result['result']}")
    # print(f"\n**Sources:**")
    # for i, doc in enumerate(result['source_documents'], 1):
    #     print(f"{i}. {doc.page_content[:200]}...")

    return result


### ask questions on the provided file like shown below

In [ ]:
questions = [
    "What is the main topic of this document?",
    "Can you summarize the key points?",
    # "What specific music are mentioned in the results? and list the selectors like css class or id",
    # "is there any song currently being played right now? what is the state"
]

for q in questions:
    ask_question(q)
    print("-" * 80)

/tmp/ipython-input-3686700383.py:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"query": question})




> Entering new RetrievalQA chain...

> Finished chain.

**Question:** What is the main topic of this document?

**Answer:** The main topic of this document appears to be the preprocessing and analysis of large webtext corpora, specifically in the context of natural language processing (NLP) and computational linguistics.
--------------------------------------------------------------------------------


> Entering new RetrievalQA chain...

> Finished chain.

**Question:** Can you summarize the key points?

**Answer:** Based on the provided context, here are the key points:

1. A test for evaluating grammar, reading comprehension, and writing style is mentioned, but the details are not fully clear.
2. The test has 3 sections, each timed, and should take 50 minutes to complete.
3. To pass the test, a candidate must score 90% on the first section and an average score of 4 on the second and third sections.
4. There are four tests in total, but the details of the other three tests are not 

In [ ]:
questions = [
    "what are the limitations and ethical considerations?",
    # "how can i play the song? give me an id for the element to play the song"
]

for q in questions:
    ask_question(q)
    print("-" * 80)



> Entering new RetrievalQA chain...

> Finished chain.

**Question:** what are the limitations and ethical considerations?

**Answer:** According to the provided context, the limitations and ethical considerations of Llama 2 are:

1. **Unpredictable outputs**: The model's potential outputs cannot be predicted in advance, and it may produce inaccurate or objectionable responses to user prompts.
2. **Limited testing**: Testing has been conducted only in English, and it has not covered all scenarios, so it's possible that the model may not perform well in certain situations.
3. **Knowledge updates**: The model's knowledge is frozen at the time of pretraining, so it will not have access to new information or updates after that point.
4. **Non-factual generation**: The model may generate non-factual information, such as unqualified advice.
5. **Hallucinations**: The model may "hallucinate" or generate information that is not based on actual knowledge.
6. **Limited understanding of real-wo

# Task
Create a Gradio GUI chat interface for the existing `qa_chain`.

In [ ]:
%pip install -q gradio

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

## Initialize the knowledge base

### Subtask:
Set up the knowledge base to store the conversation history.


**Reasoning**:
Initialize a `ConversationBufferMemory` object with the specified parameters to store the conversation history.



In [ ]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

/tmp/ipython-input-515166408.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


## Modify the chat function

### Subtask:
Update the chat function to incorporate the knowledge base and maintain the conversation history.


In [ ]:
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory
)

def respond(message, history):
    """
    Responds to a user message using the conversational QA chain.

    Args:
        message: The user's input message.
        history: The chat history (used by the memory object).

    Returns:
        The answer from the conversational QA chain.
    """
    result = conversation_chain({"question": message})
    return result['answer']

In [ ]:
import gradio as gr
iface = gr.ChatInterface(fn=respond, title="Document QA Chatbot with History")

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


## Test the chat interface

### Subtask:
Test the chat interface to ensure it remembers previous queries.


**Reasoning**:
Launch the Gradio interface to test the chat interface's ability to remember previous queries.



In [ ]:
iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e35bb9f38c5fab6284.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Notebook Summary

This notebook demonstrates how to build a Document Question Answering (QA) chatbot using LangChain, NVIDIA embeddings, and FAISS for efficient retrieval from a large document. It specifically focuses on using a PDF document ("llama2_paper.pdf") as the knowledge base.

Here's a breakdown of the key steps:

1.  **Setup:** Installs necessary libraries (`langchain`, `langchain-nvidia-ai-endpoints`, `langchain_community`, `unstructured`, `faiss-cpu`, `unstructured[pdf]`) and sets the NVIDIA API key.
2.  **Document Loading:** Loads the content of "llama2_paper.pdf" using `UnstructuredFileLoader`.
3.  **Text Splitting:** Splits the loaded document into smaller, manageable chunks using `RecursiveCharacterTextSplitter` to prepare them for embedding and indexing.
4.  **Embedding and Vector Store Creation:** Initializes NVIDIA embeddings and creates a FAISS vector store from the document chunks. This allows for efficient similarity search.
5.  **Retriever Setup:** Configures a retriever to fetch the most relevant document chunks based on a query.
6.  **Language Model Initialization:** Initializes a conversational language model (`ChatNVIDIA`) for generating responses.
7.  **RetrievalQA Chain:** Creates a `RetrievalQA` chain to combine the retriever and the language model for answering questions based on the document content.
8.  **Question Answering Function:** Defines a function `ask_question` to take a question, use the `qa_chain` to find the answer, and print the question and answer.
9.  **Testing the QA Chain:** Tests the `ask_question` function with sample questions.
10. **Gradio Chat Interface:** Sets up a Gradio chat interface using `gr.ChatInterface` and a `ConversationalRetrievalChain` with `ConversationBufferMemory` to build a chatbot that can answer questions about the document and maintain conversation history.
11. **Launching the Interface:** Launches the Gradio interface to interact with the chatbot.

The notebook demonstrates a complete workflow for building a document QA system with conversational capabilities.